In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ── Daten ──────────────────────────────────────────────────────────────────────
architectures = [r"$\mathrm{Cla}_1$", r"$\mathrm{Cla}_2$", r"$\mathrm{Cla}_3$", r"$\mathrm{Cla}_4$"]
arch_keys = ["Cla_1", "Cla_2", "Cla_3", "Cla_4"]
classes = ["Ball", "Elfmeterpunkt", "Linienkreuzung"]
n_dist_vals = [0, 1]

# Struktur: arch_key → n_dist → class → Wert
ap_values = {
    "Cla_1": {
        0: {"Ball": 0.80, "Elfmeterpunkt": 0.72, "Linienkreuzung": 0.65},
        1: {"Ball": 0.84, "Elfmeterpunkt": 0.76, "Linienkreuzung": 0.70},
    },
    "Cla_2": {
        0: {"Ball": 0.85, "Elfmeterpunkt": 0.77, "Linienkreuzung": 0.71},
        1: {"Ball": 0.88, "Elfmeterpunkt": 0.81, "Linienkreuzung": 0.75},
    },
    "Cla_3": {
        0: {"Ball": 0.89, "Elfmeterpunkt": 0.83, "Linienkreuzung": 0.77},
        1: {"Ball": 0.92, "Elfmeterpunkt": 0.86, "Linienkreuzung": 0.81},
    },
    "Cla_4": {
        0: {"Ball": 0.91, "Elfmeterpunkt": 0.85, "Linienkreuzung": 0.79},
        1: {"Ball": 0.93, "Elfmeterpunkt": 0.88, "Linienkreuzung": 0.83},
    },
}

euc_values = {
    "Cla_1": {
        0: {"Ball": 13.5, "Elfmeterpunkt": 16.2, "Linienkreuzung": 19.8},
        1: {"Ball": 11.2, "Elfmeterpunkt": 14.0, "Linienkreuzung": 17.3},
    },
    "Cla_2": {
        0: {"Ball": 11.8, "Elfmeterpunkt": 14.5, "Linienkreuzung": 17.9},
        1: {"Ball": 9.8, "Elfmeterpunkt": 12.3, "Linienkreuzung": 15.6},
    },
    "Cla_3": {
        0: {"Ball": 10.2, "Elfmeterpunkt": 12.8, "Linienkreuzung": 15.4},
        1: {"Ball": 8.5, "Elfmeterpunkt": 10.9, "Linienkreuzung": 13.2},
    },
    "Cla_4": {
        0: {"Ball": 9.1, "Elfmeterpunkt": 11.4, "Linienkreuzung": 14.0},
        1: {"Ball": 7.8, "Elfmeterpunkt": 9.8, "Linienkreuzung": 12.1},
    },
}

# ── Y-Achsen Limits ────────────────────────────────────────────────────────────
y_max_ap  = 1.0
y_max_euc = max(
    euc_values[arch_key][n][cls]
    for arch_key in arch_keys
    for n in n_dist_vals
    for cls in classes
) * 1.2

# ── Gemeinsame Layoutparameter ─────────────────────────────────────────────────
x = np.arange(len(n_dist_vals))
n_classes = len(classes)
bar_width = 0.22
offsets = np.linspace(-(n_classes - 1) / 2, (n_classes - 1) / 2, n_classes) * bar_width
colors = ["#0072B2", "#E69F00", "#009E73"]
mean_color = "#CC79A7"


# ── Hilfsfunktionen ────────────────────────────────────────────────────────────
def style_ax(ax):
    ax.set_xticks(x)
    ax.set_xticklabels([n for n in n_dist_vals], fontsize=11)
    ax.set_xlabel(r"$n_{\mathrm{dist}}$", fontsize=11)
    ax.yaxis.grid(True, linestyle="--", alpha=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)


def draw_bars(ax, data, fmt=".2f"):
    """data: n_dist → class → Wert"""
    for cls, color, offset in zip(classes, colors, offsets, strict=True):
        vals = [data[n][cls] for n in n_dist_vals]
        bars = ax.bar(
            x + offset,
            vals,
            width=bar_width,
            label=cls,
            color=color,
            edgecolor="white",
            linewidth=0.6,
            zorder=3,
        )
        for bar in bars:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.01,
                f"{bar.get_height():{fmt}}",
                ha="center",
                va="bottom",
                fontsize=10,
                color="#333333",
            )


def draw_mean_segments(ax, data, label):
    """Mittlere horizontale Segmente pro n_dist-Gruppe"""
    for xi, n in zip(x, n_dist_vals, strict=True):
        mv = np.mean([data[n][cls] for cls in classes])
        ax.hlines(
            mv,
            xi - 0.35,
            xi + 0.35,
            colors=mean_color,
            linewidth=2.5,
            linestyle="--",
            zorder=4,
            label=label if xi == 0 else "",
        )
        ax.text(xi + 0.37, mv, f"{mv:.2f}", ha="left", va="center", fontsize=10, color=mean_color)


# ── Ein PDF pro Architektur ────────────────────────────────────────────────────
for arch_key, arch_label in zip(arch_keys, architectures, strict=True):
    # — AP / mAP ——————————————————————————————————————————————————————————————
    fig, ax1 = plt.subplots(figsize=(7, 5))

    ax1.text(
        0.85,
        0.9,  # Mitte des Plots
        arch_label,
        transform=ax1.transAxes,
        fontsize=40,
        color="grey",
        alpha=0.35,  # Transparenz – höher = dunkler
        ha="center",
        va="center",
        rotation=0,  # optional: leicht gedreht
        zorder=0,  # hinter allen anderen Elementen
    )

    draw_bars(ax1, ap_values[arch_key], fmt=".2f")
    ax1.set_ylim(0, 1.15)
    draw_mean_segments(ax1, ap_values[arch_key], "Ø Gesamt")
    style_ax(ax1)
    ax1.set_ylabel("AP / mAP", fontsize=11)
    # ax1.set_title(f"AP / mAP – {arch_label}", fontsize=12)
    ax1.legend(loc="lower right", framealpha=0.9, fontsize=9)
    plt.tight_layout()
    plt.savefig(f"../../plots/n_dist/{arch_key}_ap.pdf")
    # plt.close()

    # — Euklidischer Fehler ————————————————————————————————————————————————————
    fig, ax2 = plt.subplots(figsize=(7, 5))

    ax2.text(
        0.85,
        0.9,  # Mitte des Plots
        arch_label,
        transform=ax2.transAxes,
        fontsize=40,
        color="grey",
        alpha=0.35,  # Transparenz – höher = dunkler
        ha="center",
        va="center",
        rotation=0,  # optional: leicht gedreht
        zorder=0,  # hinter allen anderen Elementen
    )

    ymax = max(euc_values[arch_key][n][cls] for n in n_dist_vals for cls in classes) * 1.2
    draw_bars(ax2, euc_values[arch_key], fmt=".1f")
    ax2.set_ylim(0, y_max_euc + 0.15)
    draw_mean_segments(ax2, euc_values[arch_key], "Ø Gesamt")
    style_ax(ax2)
    ax2.set_ylabel(r"$\bar{e}$ [px]", fontsize=11)
    # ax2.set_title(f"Euklidischer Fehler – {arch_label}", fontsize=12)
    ax2.legend(loc="lower right", framealpha=0.9, fontsize=9)
    plt.tight_layout()
    plt.savefig(f"../../plots/n_dist/{arch_key}_euc.pdf")
    # plt.close()

    print(f"Gespeichert: {arch_key}_metriken.pdf")

plt.show()